In [ ]:
import csv
import math
import matplotlib.pyplot as plt
import numpy as np

from utils import (
    read_intensities, 
    read_kmer_positions,
    get_all_kmer_wildcards,
    get_sequence_from_fasta,
    match_all_kmers_to_wildcards,
    regression_driver
)

genome =  "../../../../../d/OneDrive - McGill University/repos/hg38_ucsc.fa"
intensities = read_intensities("./data/ENCFF003ZRP_ENCFF274YGF.ChIPBM.tf.txt")
kmers = read_kmer_positions("./data/ENCFF274YGF_K562_DNase-seq_Crawford_GRCh38.ChIPBM.array.txt")
dhs = 1
region = "chr6:41071098-41071113"
kmer_size = 8
num_random = 1000
mode="neg-binomial"



In [ ]:
region = "chr5:1295220-1295260"
kmer_size = 8
num_random = 1000
mode="neg-binomial"

used_regions = set(kmers.keys())

chrom, coords = region.split(":")
start, end = map(int, coords.split("-"))
region_seq = get_sequence_from_fasta(chrom, start, end, genome)

rows = []

for i in range(len(region_seq) - kmer_size + 1):
    kmer_seq = region_seq[i:i + kmer_size]
    motif_start = start + i
    # print(f"\n[Window {i}] kmer_seq = {kmer_seq}")

    # === Step 2: Generate all possible wildcards at every position ===
    wildcard_variants = get_all_kmer_wildcards(kmer_seq)
    # print("  Wildcards:")
    # for pos, wildcard in wildcard_variants.items():
        # print(f"    Position {pos}: {wildcard}")

    # === Step 3: Match wildcards to genome-wide k-mers ===
    matched = match_all_kmers_to_wildcards(kmers, wildcard_variants)

    # print("  Matched wildcard positions:")
    for motif_pos in matched:
        for allele in matched[motif_pos]:
            count = len(matched[motif_pos][allele])
            # print(f"    Position {motif_pos}, Allele {allele}: {count} matches")

    # === Step 4: Run regression for each wildcard position and allele ===
    for motif_pos in matched:
        for allele in matched[motif_pos]:
            matched_block = {
                motif_pos: {
                    motif_pos: {
                        allele: matched[motif_pos][allele]
                    }
                }
            }

            try:
                aff_stats = regression_driver(
                    matched_block,
                    probe_intensities=intensities,
                    exclude_ref_allele=False,
                    dhs=dhs,
                    mode=mode
                )
            except Exception as e:
                print(f"[Warning] Regression failed for Window {i}, Pos {motif_pos}, Allele {allele}: {e}")
                aff_stats = {allele: (math.nan, None, math.nan)}

            coef, _, pval = aff_stats.get(allele, (math.nan, None, math.nan))
            absolute_pos = i + motif_pos

            wildcard_kmer = wildcard_variants[motif_pos]
            filled_kmer = list(wildcard_kmer)
            filled_kmer[motif_pos] = allele
            filled_kmer = ''.join(filled_kmer)

            rows.append([
                wildcard_kmer,
                filled_kmer,
                i,
                motif_pos,
                "AFF",
                allele,
                coef,
                pval,
                absolute_pos
            ])


In [ ]:
import pandas as pd
df = pd.DataFrame(rows, columns=[
    "wildcard_kmer", "filled_kmer", "window_index", "snp_index", "type", "allele", "coef", "pval", "absolute_position"
    ])

# Print aff_rows
aff_rows = df[df["type"] == "AFF"].copy()
print(aff_rows)

In [1]:
def plot_aff_motif_effects(rows):

    df = pd.DataFrame(rows, columns=[
        "wildcard_kmer", "filled_kmer", "window_index", "snp_index",
        "type", "allele", "coef", "pval", "absolute_pos"
    ])

    # Filter for AFF rows only
    df = df[df["type"] == "AFF"].copy()
    df["coef"] = pd.to_numeric(df["coef"], errors="coerce")
    df["pval"] = pd.to_numeric(df["pval"], errors="coerce")
    df["-log10(pval)"] = -np.log10(df["pval"])

    # Make x-axis 1-based instead of 0-based
    df["absolute_position"] = df["window_index"] + df["snp_index"] + 1
    region_length = df["absolute_position"].max()

    allele_colors = {
        "A": "black",
        "C": "red",
        "G": "green",
        "T": "blue"
    }

    fig_width = max(12, region_length * 0.15)
    fig, axs = plt.subplots(2, 1, figsize=(fig_width, 6), sharex=True)

    metrics = ["coef", "-log10(pval)"]

    for metric, ax in zip(metrics, axs):
        for _, row in df.iterrows():
            x = int(row["absolute_position"])
            y = row[metric]
            allele = row["allele"]
            color = allele_colors.get(allele, "gray")
            if pd.notna(y):
                ax.text(x, y, allele, color=color, fontsize=12, ha="center", va="center", fontweight="bold")

        # Y-axis limits with padding
        ymin = df[metric].min()
        ymax = df[metric].max()
        yrange = ymax - ymin if ymax != ymin else 1
        padding = yrange * 0.1
        ax.set_ylim(ymin - padding, ymax + padding)
        ax.set_ylabel(metric)
        ax.grid(True, linestyle="--", alpha=0.4)

    axs[1].set_xlabel("Genomic Position")
    axs[0].set_title("Motif Scan: Coefficients and -log10(p-values)")

    # X-axis ticks
    step = 10 if region_length > 80 else 5 if region_length > 40 else 1
    axs[1].set_xticks(range(1, region_length + 1, step))

    # Prevent overflow
    axs[1].set_xlim(0.5, region_length + 0.5)

    plt.tight_layout()
    plt.show()

plot_aff_motif_effects(rows)

NameError: name 'rows' is not defined

In [1]:
import csv
import math
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

from utils import (
    read_intensities, 
    read_kmer_positions,
    get_all_kmer_wildcards,
    get_sequence_from_fasta,
    match_all_kmers_to_wildcards,
    regression_driver,
    print_rows_as_tsv,
    plot_aff_motif_effects
)

genome =  "../../../../../d/OneDrive - McGill University/repos/hg38_ucsc.fa"
intensities = read_intensities("data/mcf7_gabpa/GABPA_MCF7_probeIntensity.bed")
kmers = read_kmer_positions("data/mcf7_gabpa/GABPA_Array_ATAC.txt")
dhs = 1
region = "chr5:1295105-1295140"
# region = "chr6:1295220-1295228"
kmer_size = 8
num_random = 1000
mode="neg-binomial"

/home/aki/miniconda3/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
def filter_unique_kmer_hits(kmer_positions):
    """
    Removes k-mer entries that map to the same region more than once (i.e., with .1, .2, etc.).
    Keeps only base regions that appear exactly once.
    """
    filtered = {}

    for kmer, region_dict in kmer_positions.items():
        # Count base regions
        base_counts = {}
        for region_id in region_dict:
            base = region_id.split(".")[0]
            base_counts[base] = base_counts.get(base, 0) + 1

        # Now filter to keep only base regions with count == 1
        filtered[kmer] = {
            region_id: offset
            for region_id, offset in region_dict.items()
            if base_counts[region_id.split(".")[0]] == 1
        }

    return filtered


kmers = filter_unique_kmer_hits(kmers)


In [3]:
import re
from collections import defaultdict

def scan_motif_kmers(region_seq, kmer_size, kmer_positions):
    """
    Slides a k-mer window across the sequence and builds:
      - allele_region_offsets[motif_pos][snv_index][allele][region_id] = offset
      - allele_matched_kmers[motif_pos][snv_index][allele] = list of (wildcard_kmer, matched_kmer)

    Returns:
        (allele_region_offsets, allele_matched_kmers)
    """
    from collections import defaultdict
    import re

    allele_region_offsets = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    allele_matched_kmers = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    for motif_pos in range(len(region_seq) - kmer_size + 1):
        kmer = region_seq[motif_pos:motif_pos + kmer_size]

        for snv_index in range(kmer_size):
            # Create wildcarded k-mer
            wildcard_kmer = list(kmer)
            wildcard_kmer[snv_index] = "."
            wildcard_kmer = "".join(wildcard_kmer)

            for base in "ACGT":
                filled_kmer = list(kmer)
                filled_kmer[snv_index] = base
                filled_kmer = "".join(filled_kmer)

                if filled_kmer in kmer_positions:
                    for region_id, offset in kmer_positions[filled_kmer].items():
                        allele_region_offsets[motif_pos][snv_index][base][region_id] = offset
                        allele_matched_kmers[motif_pos][snv_index][base].append((wildcard_kmer, filled_kmer))

    return allele_region_offsets, allele_matched_kmers


chrom, coords = region.split(":")
start, end = map(int, coords.split("-"))
region_seq = get_sequence_from_fasta(chrom, start, end, genome)

allele_region_offsets, allele_matched_kmers = scan_motif_kmers(
    region_seq=region_seq,
    kmer_size=kmer_size,
    kmer_positions=kmers
)

In [ ]:
# Sanity check
def summarize_matched_kmers(allele_matched_kmers):
    print("Summary of matched k-mers by motif position and SNV index:\n")
    
    for motif_pos in sorted(allele_matched_kmers.keys()):
        for snv_index in sorted(allele_matched_kmers[motif_pos].keys()):
            print(f"Motif position {motif_pos}, SNV index {snv_index}")
            for allele in "ACGT":
                matches = allele_matched_kmers[motif_pos][snv_index][allele]
                count = len(matches)
                if count > 0:
                    print(f"  Allele {allele}: {count} matches")
                    print(f"    Example wildcard -> filled: {matches[0]}")
            print("-" * 40)
            
summarize_matched_kmers(allele_matched_kmers)

In [ ]:
def run_snv_regression(
    motif_pos,
    snv_index,
    allele_region_offsets,
    allele_matched_kmers,
    intensities,
    model_type="nb",
    include_sl=False,
):
    import statsmodels.api as sm
    import numpy as np
    import pandas as pd
    import math

    rows = []

    alleles = allele_region_offsets[motif_pos][snv_index]
    all_regions = set()
    for region_dict in alleles.values():
        all_regions.update(region_dict.keys())

    design = []
    y_values = []
    region_list = []
    lp_vals = []
    sl_vals = []

    for region in all_regions:
        row = {a: 0 for a in alleles}
        offset_found = False
        offset = None
        for a in alleles:
            if region in alleles[a]:
                row[a] = 1
                if not offset_found:
                    offset = allele_region_offsets[motif_pos][snv_index][a][region]
                    offset_found = True
        if region in intensities and offset_found:
            chrom, coords = region.split(":")
            start, end = map(int, coords.split("-"))
            region_len = end - start
            lp_val = offset / region_len if region_len > 0 else 0.5
            sl_val = region_len

            design.append(row)
            y_values.append(intensities[region])
            region_list.append(region)
            lp_vals.append(lp_val)
            sl_vals.append(sl_val)

    if not design:
        return []

    X = pd.DataFrame(design)
    y = pd.Series(y_values, index=region_list)
    X.index = y.index
    X["lp"] = lp_vals
    if include_sl:
        X["sl"] = sl_vals

    X_const = sm.add_constant(X)

    if model_type == "ols":
        model = sm.OLS(np.log1p(y), X_const)
    else:
        model = sm.GLM(y, X_const, family=sm.families.NegativeBinomial())

    results = model.fit()

    for allele in alleles:
        coef = results.params.get(allele, np.nan)
        pval = results.pvalues.get(allele, np.nan)

        for wildcard_kmer, filled_kmer in set(allele_matched_kmers[motif_pos][snv_index][allele]):
            row = [
                wildcard_kmer,
                filled_kmer,
                motif_pos,
                snv_index,
                "AFF",
                allele,
                coef,
                pval,
                motif_pos + snv_index  # absolute position
            ]
            rows.append(row)

    return rows

all_rows = []

for motif_pos in sorted(allele_region_offsets):
    for snv_index in sorted(allele_region_offsets[motif_pos]):
        all_rows.extend(run_snv_regression(
            motif_pos, snv_index,
            allele_region_offsets, allele_matched_kmers,
            intensities, model_type="nb", include_sl = True
        ))

plot_aff_motif_effects(all_rows, save_path="newest2.png)

NameError: name 'allele_region_offsets' is not defined